# 00. Environment Setup & Verification

This notebook verifies that your environment is correctly set up for the lens correction project.

**Goals**:
1. Verify Python version
2. Check GPU availability
3. Test package imports
4. Verify data directory structure
5. Display system information

## 1. Python Environment

In [1]:
import sys
print(f"Python version: {sys.version}")
print(f"Python executable: {sys.executable}")

Python version: 3.11.9 (main, Aug 14 2024, 04:18:20) [MSC v.1929 64 bit (AMD64)]
Python executable: c:\Users\Robbinhood\OneDrive\Desktop\VS Code Projects\AutoHDR Project\.venv\Scripts\python.exe


## 2. GPU Check

In [2]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU count: {torch.cuda.device_count()}")
    print(f"Current GPU: {torch.cuda.current_device()}")
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    
    # Memory info
    mem_allocated = torch.cuda.memory_allocated(0) / 1024**3
    mem_reserved = torch.cuda.memory_reserved(0) / 1024**3
    mem_total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    
    print(f"\nMemory allocated: {mem_allocated:.2f} GB")
    print(f"Memory reserved: {mem_reserved:.2f} GB")
    print(f"Total memory: {mem_total:.2f} GB")
else:
    print("⚠️  No GPU detected. Training will be slow on CPU.")

PyTorch version: 2.12.0.dev20260225+cu128
CUDA available: True
CUDA version: 12.8
GPU count: 1
Current GPU: 0
GPU name: NVIDIA GeForce RTX 5060 Ti

Memory allocated: 0.00 GB
Memory reserved: 0.00 GB
Total memory: 15.93 GB


## 3. Package Imports

In [3]:
# Core ML libraries
import torch
import torchvision
import pytorch_lightning as pl
import timm

# Computer vision
import cv2
import albumentations as A
from kornia import filters

# Data science
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Utilities
from pathlib import Path
from tqdm.auto import tqdm
import yaml

print("✅ All imports successful!")

W0225 13:26:12.546000 22360 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


✅ All imports successful!


## 4. Version Check

In [4]:
versions = {
    "Python": sys.version.split()[0],
    "PyTorch": torch.__version__,
    "Torchvision": torchvision.__version__,
    "PyTorch Lightning": pl.__version__,
    "timm": timm.__version__,
    "OpenCV": cv2.__version__,
    "Albumentations": A.__version__,
    "NumPy": np.__version__,
    "Pandas": pd.__version__,
}

print("Package Versions:")
for package, version in versions.items():
    print(f"  {package:20s}: {version}")

Package Versions:
  Python              : 3.11.9
  PyTorch             : 2.12.0.dev20260225+cu128
  Torchvision         : 0.26.0.dev20260221+cu128
  PyTorch Lightning   : 2.6.1
  timm                : 1.0.25
  OpenCV              : 4.13.0
  Albumentations      : 2.0.8
  NumPy               : 2.4.2
  Pandas              : 3.0.1


## 5. Directory Structure

In [5]:
# Check project root
project_root = Path("../").resolve()
print(f"Project root: {project_root}")

# Check expected directories
expected_dirs = [
    "data",
    "configs",
    "notebooks",
    "src",
    "scripts",
    "tests",
    "outputs",
]

print("\nDirectory structure:")
for dir_name in expected_dirs:
    dir_path = project_root / dir_name
    exists = "✅" if dir_path.exists() else "❌"
    print(f"  {exists} {dir_name}/")

Project root: C:\Users\Robbinhood\OneDrive\Desktop\VS Code Projects\AutoHDR Project

Directory structure:
  ✅ data/
  ✅ configs/
  ✅ notebooks/
  ✅ src/
  ✅ scripts/
  ✅ tests/
  ✅ outputs/


## 6. Data Availability

In [6]:
data_dir = project_root / "data"

# Check for data subdirectories
train_dir = data_dir / "train"
test_dir = data_dir / "test"

print("Data availability:")
if train_dir.exists():
    n_train = len(list(train_dir.glob("*")))
    print(f"  ✅ Training data: {n_train} items")
else:
    print("  ⚠️  Training data not found. Run data download script.")

if test_dir.exists():
    n_test = len(list(test_dir.glob("*.jpg")))
    print(f"  ✅ Test data: {n_test} images")
else:
    print("  ⚠️  Test data not found. Run data download script.")

Data availability:
  ✅ Training data: 46238 items
  ✅ Test data: 1000 images


## 7. Test Computation

In [7]:
# Simple GPU test
if torch.cuda.is_available():
    device = torch.device("cuda")
    
    # Create random tensors
    x = torch.randn(1000, 1000, device=device)
    y = torch.randn(1000, 1000, device=device)
    
    # Matrix multiplication
    %timeit -n 10 -r 3 z = torch.mm(x, y)
    
    print("✅ GPU computation working!")
else:
    print("⚠️  Skipping GPU test (no GPU available)")

The slowest run took 1315.65 times longer than the fastest. This could mean that an intermediate result is being cached.
14.1 ms ± 19.9 ms per loop (mean ± std. dev. of 3 runs, 10 loops each)
✅ GPU computation working!


## 8. Configuration Files

In [8]:
configs_dir = project_root / "configs"

print("Available configurations:")
for config_file in configs_dir.glob("*.yaml"):
    print(f"  ✅ {config_file.name}")
    
# Load and display a config
baseline_config = configs_dir / "train_baseline.yaml"
if baseline_config.exists():
    with open(baseline_config) as f:
        config = yaml.safe_load(f)
    print(f"\nBaseline config preview:")
    print(f"  Model: {config['model']['encoder']}")
    print(f"  Batch size: {config['training']['batch_size']}")
    print(f"  Epochs: {config['training']['num_epochs']}")

Available configurations:
  ✅ inference.yaml
  ✅ train_advanced.yaml
  ✅ train_baseline.yaml

Baseline config preview:
  Model: efficientnet-b3
  Batch size: 8
  Epochs: 50


## 9. Summary

In [9]:
print("=" * 60)
print("ENVIRONMENT SETUP SUMMARY")
print("=" * 60)
print(f"✅ Python {sys.version.split()[0]}")
print(f"✅ PyTorch {torch.__version__} with {'GPU (CUDA)' if torch.cuda.is_available() else 'CPU only'}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)} ({mem_total:.1f} GB)")
print(f"✅ All packages imported successfully")
print(f"✅ Project structure verified")
print("\n🚀 Ready to start lens correction project!")
print("\nNext step: Download data and run 01_eda_and_analysis.ipynb")

ENVIRONMENT SETUP SUMMARY
✅ Python 3.11.9
✅ PyTorch 2.12.0.dev20260225+cu128 with GPU (CUDA)
✅ GPU: NVIDIA GeForce RTX 5060 Ti (15.9 GB)
✅ All packages imported successfully
✅ Project structure verified

🚀 Ready to start lens correction project!

Next step: Download data and run 01_eda_and_analysis.ipynb
